# Pulmonary fibrosis simulation — interactive runnerRuns the coupled model from[`Danpc11/lung-nematic`](https://github.com/Danpc11/lung-nematic): alveolararchitecture, the AT2 → KRT8+ → AT1 epithelial state machine, surfactant-drivencollapse, breathing with strain redistribution, and a confined mesenchyme thatfills collapsed alveoli.Run the cells in order. Every parameter is a form field — no code editing needed.**Before trusting any number**, read the caveats in[`simulations/alveolar/README.md`](https://github.com/Danpc11/lung-nematic/blob/main/simulations/alveolar/README.md).Two results in this model are negative, and the defect counts are close to acounting-noise floor.

In [ ]:
#@title 1 · Setup — clone the repository and install dependencies { display-mode: "form" }import os, subprocess, sysREPO = "https://github.com/Danpc11/lung-nematic.git"  #@param {type:"string"}BRANCH = "main"  #@param {type:"string"}ROOT = "/content/lung-nematic"if os.path.isdir(ROOT):    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=False)    subprocess.run(["git", "-C", ROOT, "reset", "--hard", f"origin/{BRANCH}"], check=False)else:    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, ROOT], check=True)subprocess.run([sys.executable, "-m", "pip", "install", "-q",                "imageio", "imageio-ffmpeg"], check=True)# Colab keeps imported modules in memory, so a fresh pull is invisible until# the cached copies are dropped. Purge before importing.for name in [n for n in list(sys.modules) if n.startswith("simulations")]:    del sys.modules[name]if ROOT not in sys.path:    sys.path.insert(0, ROOT)from simulations.alveolar import AlveolarConfig, run_and_record_coupledimport numpy as np, pandas as pd, matplotlib.pyplot as pltcommit = subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"],                        capture_output=True, text=True).stdout.strip()print(f"repository ready at commit {commit}")print(f"{len(AlveolarConfig.__dataclass_fields__)} parameters available")

## ParametersThree groups. Defaults reproduce the two-year progressive run in the repository.

In [ ]:
#@title 2 · Scenario — the lesion and how long to run { display-mode: "form" }#@markdown Total simulated time and how long the AT2→AT1 block lasts.years_to_simulate = 2.0  #@param {type:"slider", min:0.25, max:5, step:0.25}injury_duration_months = 3.3  #@param {type:"slider", min:0, max:24, step:0.1}timestep_hours = 2.0  #@param [1.0, 2.0, 4.0] {type:"raw"}#@markdown How completely differentiation fails inside the lesion#@markdown (1.0 = no block, 0.05 = severe), and how hard AT2 are pushed to try.repair_failure_factor = 0.10  #@param {type:"slider", min:0.02, max:1.0, step:0.01}injury_activation_boost = 10.0  #@param {type:"slider", min:1, max:30, step:1}injury_radius_um = 260.0  #@param {type:"slider", min:80, max:500, step:10}#@markdown Global clock: multiplies matrix and disease kinetics. Lower = slower#@markdown disease. Cell birth, death and motility keep their own clock.rate_scale = 0.08  #@param {type:"slider", min:0.01, max:1.0, step:0.01}#@markdown Domain size. Larger is more alveoli but slower.domain_um = 1400.0  #@param {type:"slider", min:800, max:2200, step:100}random_seed = 0  #@param {type:"integer"}scenario = dict(    total_time_h=years_to_simulate * 8760.0,    injury_duration_h=injury_duration_months * 730.0,    dt_h=timestep_hours,    repair_failure_factor=repair_failure_factor,    injury_activation_boost=injury_activation_boost,    injury_radius_um=injury_radius_um,    rate_scale=rate_scale,    width_um=domain_um, height_um=domain_um,    seed=random_seed,)print(f"{years_to_simulate:.2f} years, block lifted at month {injury_duration_months:.1f}")

In [ ]:
#@title 3 · Biology — the two feedback loops, breathing and cell turnover { display-mode: "form" }#@markdown **Epithelial loop.** How strongly profibrotic signal keeps cells in#@markdown the KRT8+ state, and how fast aberrant cells are cleared. Low#@markdown clearance = apoptosis resistance.stall_promotion_strength = 25.0  #@param {type:"slider", min:0, max:40, step:0.5}repair_inhibition_strength = 3.0  #@param {type:"slider", min:0, max:10, step:0.5}aberrant_clearance_rate = 0.0005  #@param {type:"number"}aberrant_emt_rate = 0.004  #@param {type:"number"}#@markdown **Matrix loop.** Collagen deposition against turnover. Their ratio#@markdown sets the point of no return.deposition_rate_kPa_per_h = 0.16  #@param {type:"number"}degradation_rate_per_h = 0.004  #@param {type:"number"}E_act_kPa = 16.0  #@param {type:"slider", min:4, max:40, step:0.5}#@markdown **Breathing.** Tidal strain is redistributed over whatever can still#@markdown deform, so stiff and collapsed regions shed their share onto healthy#@markdown tissue. Cyclic strain also *suppresses* myofibroblast conversion.tidal_strain = 0.10  #@param {type:"slider", min:0.02, max:0.25, step:0.01}strain_protection_strength = 6.0  #@param {type:"slider", min:0, max:20, step:0.5}strain_tgfb_gain = 0.8  #@param {type:"slider", min:0, max:3, step:0.1}overstrain_threshold = 0.16  #@param {type:"slider", min:0.05, max:0.4, step:0.01}#@markdown **Mesenchyme.** Cell shape sets both the excluded volume and the#@markdown resolvable scale; death rates encode apoptosis resistance.cell_length_um = 50.0  #@param {type:"slider", min:20, max:90, step:5}cell_width_um = 11.0  #@param {type:"slider", min:5, max:25, step:1}fibroblast_death_rate = 0.002  #@param {type:"number"}myofibroblast_death_rate = 0.0004  #@param {type:"number"}matrix_immobilization = 25.0  #@param {type:"slider", min:0, max:200, step:5}#@markdown **Collapse mechanics.**tissue_recoil_Pa = 420.0  #@param {type:"slider", min:200, max:800, step:10}induration_time_h = 240.0  #@param {type:"number"}biology = dict(    stall_promotion_strength=stall_promotion_strength,    repair_inhibition_strength=repair_inhibition_strength,    aberrant_clearance_rate=aberrant_clearance_rate,    aberrant_emt_rate=aberrant_emt_rate,    deposition_rate_kPa_per_h=deposition_rate_kPa_per_h,    degradation_rate_per_h=degradation_rate_per_h,    E_act_kPa=E_act_kPa,    tidal_strain=tidal_strain,    strain_protection_strength=strain_protection_strength,    strain_tgfb_gain=strain_tgfb_gain,    overstrain_threshold=overstrain_threshold,    cell_length_um=cell_length_um, cell_width_um=cell_width_um,    fibroblast_death_rate=fibroblast_death_rate,    myofibroblast_death_rate=myofibroblast_death_rate,    matrix_immobilization=matrix_immobilization,    tissue_recoil_Pa=tissue_recoil_Pa,    induration_time_h=induration_time_h,)# Warn when the director field will be counting-noise dominated.import numpy as _npcell_area = _np.pi * 0.25 * cell_length_um * cell_width_umr_min = _np.sqrt(30 * cell_area / _np.pi)print(f"cell footprint {cell_area:.0f} um^2 -> reliable director needs a window "      f"of radius >= {r_min:.0f} um")if r_min > 28.0:    print(f"  NOTE: coarse_grain_um defaults to 28 um, below that floor. "          f"Defect counts will sit near the noise floor - see the README.")

In [ ]:
#@title 4 · Visualisation and output { display-mode: "form" }frames_per_run = 60  #@param {type:"slider", min:10, max:150, step:5}frames_per_second = 8  #@param {type:"slider", min:2, max:24, step:1}resolution_dpi = 110  #@param {type:"slider", min:60, max:200, step:10}#@markdown Right-hand panel shows where the breath goes. Turning it off roughly#@markdown halves the rendering time.show_strain_panel = True  #@param {type:"boolean"}show_defects = True  #@param {type:"boolean"}cell_transparency = 0.55  #@param {type:"slider", min:0.1, max:1.0, step:0.05}stiffness_colormap = "pink_r"  #@param ["pink_r", "inferno", "magma", "YlOrBr", "bone_r"]strain_colormap = "RdBu_r"  #@param ["RdBu_r", "coolwarm", "seismic", "PuOr"]export_mp4 = True  #@param {type:"boolean"}export_gif = True  #@param {type:"boolean"}visualisation = dict(    fps=frames_per_second, dpi=resolution_dpi,    show_strain_panel=show_strain_panel, show_defects=show_defects,    cell_alpha=cell_transparency,    stiffness_cmap=stiffness_colormap, strain_cmap=strain_colormap,    make_mp4=export_mp4, make_gif=export_gif,)print(f"{frames_per_run} frames at {frames_per_second} fps -> "      f"{frames_per_run/frames_per_second:.1f} s of video")

In [ ]:
#@title 5 · Run the simulation { display-mode: "form" }import timefrom dataclasses import replaceconfig = replace(AlveolarConfig(), **scenario, **biology)config.validate()frame_every_h = config.total_time_h / frames_per_runn_steps = int(config.total_time_h / config.dt_h)seconds_est = n_steps * 0.011 + frames_per_run * (2.3 if show_strain_panel else 1.2)print(f"{n_steps:,} steps, {frames_per_run} frames  ->  roughly "      f"{seconds_est/60:.1f} min. Starting...\n")start = time.time()def show_progress(n):    done = n / (frames_per_run + 1)    bar = "#" * int(done * 30)    print(f"\r  [{bar:<30}] {done*100:5.1f}%  {time.time()-start:5.0f}s",          end="", flush=True)outputs = run_and_record_coupled(    config, "/content/results", frame_every_h=frame_every_h,    progress=show_progress, **visualisation,)print(f"\n\nfinished in {time.time()-start:.0f} s")frame = pd.read_csv(outputs["timeseries"])final = outputs["final"]print(f"\nAt {final['time_d']/365:.2f} years:")print(f"  AT1 coverage         {final['frac_AT1']*100:5.1f} %")print(f"  aberrant basaloid    {final['frac_aberrant']*100:5.1f} %")print(f"  alveoli indurated    {final['frac_indurated']*100:5.1f} %")print(f"  mesenchymal cells    {final['n_mesenchymal']:5d} "      f"({final['n_myofibroblast']} myofibroblast)")print(f"  peak stiffness       {final['max_stiffness_kPa']:5.1f} kPa")print(f"  septal thickness     {final['mean_septal_thickness_um']:5.1f} um")print(f"  strain amplification {final['strain_amplification']:5.2f} x")

In [ ]:
#@title 6 · Watch it { display-mode: "form" }from base64 import b64encodefrom IPython.display import HTML, Image, displayif export_mp4 and "mp4" in outputs:    data = b64encode(open(outputs["mp4"], "rb").read()).decode()    display(HTML(f'''<video width="960" controls loop>        <source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'''))elif export_gif and "gif" in outputs:    display(Image(filename=outputs["gif"]))else:    print("Enable export_mp4 or export_gif in cell 4 to preview.")

In [ ]:
#@title 7 · Time courses { display-mode: "form" }months = frame["time_d"] / 30.4figure, axes = plt.subplots(2, 2, figsize=(13, 7.5))ax = axes[0, 0]ax.plot(months, frame["frac_AT1"] * 100, label="AT1", color="#009E73")ax.plot(months, frame["frac_aberrant"] * 100, label="aberrant basaloid", color="#D55E00")ax.plot(months, frame["frac_KRT8"] * 100, label="KRT8+ transitional", color="#E69F00")ax.plot(months, frame["frac_denuded"] * 100, label="denuded", color="#999999")ax.set_ylabel("% of epithelium"); ax.set_title("Epithelial composition"); ax.legend(fontsize=8)ax = axes[0, 1]ax.plot(months, frame["frac_open"] * 100, label="open", color="#0072B2")ax.plot(months, frame["frac_collapsed"] * 100, label="collapsed", color="#999999")ax.plot(months, frame["frac_indurated"] * 100, label="indurated", color="#8C6D5B")ax.set_ylabel("% of alveoli"); ax.set_title("Alveolar fate"); ax.legend(fontsize=8)ax = axes[1, 0]ax.plot(months, frame["n_mesenchymal"], label="mesenchymal", color="#0072B2")ax.plot(months, frame["n_myofibroblast"], label="myofibroblast", color="#D55E00")ax.set_ylabel("cells"); ax.set_xlabel("months"); ax.set_title("Mesenchyme"); ax.legend(fontsize=8)ax = axes[1, 1]ax.plot(months, frame["max_stiffness_kPa"], color="#8C4A1E", label="peak stiffness (kPa)")ax.plot(months, frame["mean_septal_thickness_um"], color="#7F3B08", label="septal thickness (um)")twin = ax.twinx()twin.plot(months, frame["strain_amplification"], color="#56B4E9", ls="--",          label="strain amplification")twin.set_ylabel("strain / normal", color="#56B4E9")ax.set_xlabel("months"); ax.set_title("Matrix and mechanics"); ax.legend(fontsize=8, loc="upper left")for row in axes:    for cell in row:        cell.axvline(config.injury_duration_h / 730.0, color="k", ls=":", lw=1)figure.suptitle("dotted line = the AT2->AT1 block is lifted", fontsize=9, y=1.00)figure.tight_layout(); plt.show()

In [ ]:
#@title 8 · Export everything { display-mode: "form" }import json, shutil, zipfilefrom dataclasses import asdictfrom pathlib import Pathbundle_name = "ipf_simulation_run"  #@param {type:"string"}also_include_frames = False  #@param {type:"boolean"}results = Path("/content/results")bundle = Path(f"/content/{bundle_name}.zip")with open(results / "config.json", "w") as handle:    json.dump(asdict(config), handle, indent=2)frame.to_csv(results / "timeseries.csv", index=False)with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:    for item in results.iterdir():        if item.is_file():            archive.write(item, item.name)    if also_include_frames and (results / "frames").is_dir():        for item in sorted((results / "frames").glob("*.png")):            archive.write(item, f"frames/{item.name}")size_mb = bundle.stat().st_size / 1e6print(f"{bundle.name}  ({size_mb:.1f} MB)")print("contains: video, time series, and the config that reproduces this run")try:    from google.colab import files    files.download(str(bundle))except Exception as error:    print(f"\nAutomatic download unavailable ({error}).")    print(f"Use the file browser on the left: {bundle}")

## Reproducing a run laterThe exported `config.json` holds every parameter. To rerun it:```pythonimport jsonfrom dataclasses import replacefrom simulations.alveolar import AlveolarConfig, run_and_record_coupledwith open("config.json") as handle:    config = replace(AlveolarConfig(), **json.load(handle))run_and_record_coupled(config, "results/rerun", frame_every_h=292.0)```Parameter sets backing a figure belong in `simulations/configs/` in therepository, which is version-controlled — unlike the results directory.## What this model does not doTwo dimensions only; alveolar collapse rescales a radius rather than relaxingseptal mechanics; breathing is quasi-static, so individual breaths are neverresolved; and the defect counts sit near a counting-noise floor set by cell size.The `rate_scale` default was chosen so the disease course lands near two years —it is a calibration to a clinical impression, not a fit to data.## Two things to know before reading a result**The first weeks are a transient, not biology.** `n_resident_fibroblasts`seeds 260 cells, which is above the homeostatic density of the confinedinterstitium, so the population falls sharply before the lesion drives it backup. In a two-year run it has recovered by about month 4; in a short run you mayonly be watching it relax. Either discard the opening months or lower the seed.**Defect counts are near a noise floor.** Cell 3 prints the minimum windowradius needed for a reliable director field, `sqrt(30 * A_cell / pi)`. With thedefault 50x11 um cell that is 64 um, while `coarse_grain_um` is 28 um. Belowthat floor the apparent nematic order is dominated by counting noise, and thedetected defects are not persistent objects. This is analysed in the README andis a property of the tissue geometry, not a bug to tune away.